# 隐私确认闸门 · 最简版本

先别管 Agent A / Agent B 的智能程度，这个 notebook 只做一件事：
让你亲手感受一遍 **"本地分析 → 生成清单 → 暂停等你确认 → 你说了算才继续"** 这条链路。

Agent A、Agent B 里面都是假数据，一路跑通之后，我们再一点点替换成真实逻辑。

In [2]:
from typing import TypedDict, Literal, Optional
from uuid import uuid4

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

/Users/steven/miniconda3/envs/pytorch/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## 1. 状态：整个流程里会传递的数据

注意 `agent_a_result`（完整持仓）和 `desensitized_list`（脱敏清单）是分开存的——
这一点很重要，后面你会看到只有 `desensitized_list` 才有机会离开本地。

In [3]:
class State(TypedDict, total=False):
    agent_a_result: dict
    desensitized_list: list[dict]
    user_decision: Literal["approved", "rejected"]
    agent_b_result: dict
    fusion_result: dict

## 2. 本地 Agent A（假数据）+ 生成脱敏清单

`ALLOWED` 就是最简单版本的"白名单"——先用一行代码过滤，够用了，以后再升级成严格校验。

In [4]:
def agent_a_node(state: State) -> dict:
    full_result = {
        "ticker": "600570",
        "name": "恒生电子",
        "qty": 1000,        # 持仓数量：不脱敏
        "cost": 32.5,        # 成本价：不脱敏
        "industry_tags": ["金融科技"],
    }
    ALLOWED = {"ticker", "name", "industry_tags"}
    desensitized = {k: v for k, v in full_result.items() if k in ALLOWED}
    return {"agent_a_result": full_result, "desensitized_list": [desensitized]}

## 3. 确认节点：暂停，把清单交给用户

`interrupt(...)` 一调用，图就会在这里冻结，`payload` 会被原样交还给外面调用它的代码
（也就是你后面会看到的 `result["__interrupt__"]`）。

In [5]:
def human_confirm_node(state: State) -> dict:
    decision = interrupt({
        "will_send": state["desensitized_list"],
        "will_not_send": ["持仓数量", "成本价"],
    })
    return {"user_decision": decision["action"]}

## 4. 确认之后：云端 Agent B（假数据，只吃脱敏清单）+ 融合

取消之后：什么都不做，直接结束——这一步用来证明"取消了，云端真的没被调用"。

In [6]:
def route_after_confirm(state: State) -> str:
    return "agent_b" if state["user_decision"] == "approved" else END

def agent_b_node(state: State) -> dict:
    # 注意：这里只能拿到 desensitized_list，agent_a_result 从没传进来
    tickers = [item["ticker"] for item in state["desensitized_list"]]
    return {"agent_b_result": {"summary": f"围绕 {tickers} 检索到公开新闻，情绪偏正面。"}}

def fusion_node(state: State) -> dict:
    return {"fusion_result": {"advice": "持有并关注", "basis": state["agent_b_result"]}}

## 5. 组图

In [7]:
graph = StateGraph(State)
graph.add_node("agent_a", agent_a_node)
graph.add_node("human_confirm", human_confirm_node)
graph.add_node("agent_b", agent_b_node)
graph.add_node("fusion", fusion_node)

graph.add_edge(START, "agent_a")
graph.add_edge("agent_a", "human_confirm")
graph.add_conditional_edges("human_confirm", route_after_confirm, {"agent_b": "agent_b", END: END})
graph.add_edge("agent_b", "fusion")
graph.add_edge("fusion", END)

app = graph.compile(checkpointer=MemorySaver())

## 6. 第一次运行：会在确认节点暂停

跑完这个 cell，图已经"冻结"了，往下不会再自动执行，直到你显式 resume。

In [9]:
thread_id = str(uuid4())
config = {"configurable": {"thread_id": thread_id}}

result = app.invoke({}, config=config)
result["__interrupt__"][0].value  # 这就是应该拿去渲染确认弹窗的内容

{'will_send': [{'ticker': '600570',
   'name': '恒生电子',
   'industry_tags': ['金融科技']}],
 'will_not_send': ['持仓数量', '成本价']}

## 7. 模拟用户点了"确认发送"

用同一个 `thread_id`，把用户的决定通过 `Command(resume=...)` 传回去，图会从冻结的地方精确恢复。

In [10]:
final = app.invoke(Command(resume={"action": "approved"}), config=config)
final["fusion_result"]

{'advice': '持有并关注', 'basis': {'summary': "围绕 ['600570'] 检索到公开新闻，情绪偏正面。"}}

## 8. 换一个新会话，这次模拟用户点"取消"

看看云端到底有没有被调用——`agent_b_result` 应该完全不存在。

In [11]:
thread_id_2 = str(uuid4())
config2 = {"configurable": {"thread_id": thread_id_2}}

app.invoke({}, config=config2)
final2 = app.invoke(Command(resume={"action": "rejected"}), config=config2)

print("是否有 agent_b_result：", "agent_b_result" in final2)
print("最终状态：", final2)

是否有 agent_b_result： False
最终状态： {'agent_a_result': {'ticker': '600570', 'name': '恒生电子', 'qty': 1000, 'cost': 32.5, 'industry_tags': ['金融科技']}, 'desensitized_list': [{'ticker': '600570', 'name': '恒生电子', 'industry_tags': ['金融科技']}], 'user_decision': 'rejected'}


---
### 下一步可以加什么（先跑通再说，想到了随时聊）

- 把 Agent A 换成真实的 AKShare 数据
- 把脱敏规则从"一行过滤"升级成严格 schema 校验
- 加一个 `edited_list`，让用户在确认时能删掉某个标的
- 包一层 FastAPI，让前端能真正弹出这个确认框
